# Optimización del costo para la asignación de profesores (premium)

**Problema de asignación**: ECs baratos y con disponibilidad no están quedando asignados. 
**Solución y otras incorporaciones**: 
1. Hacer una asignación por capas, para garantizar que ningun EC costoso le quite series a un EC barato. 
2. Randomizar la selección cuando tienen el mismo fee
3. Glenda (email g.vega@slangapp.com) es la única retainer y toca contenerla en 80-90h mensuales. (esto quisiera hacerlo modificandole el fee para no cambiar mucho la logica del cogido)


### Importamos paquetes

In [728]:
# si es la primera vez que se ejecuta, tenemos que instalar PuLP usando la terminal
# utilice este comando:
# pip install pulp
import pandas as pd
import numpy as np
import random
import numpy as np
import matplotlib.pyplot as plt
from statistics import mode, StatisticsError
from datetime import datetime, timedelta

In [729]:
# si queremos quitar warnings mamones!
import warnings

# Suppress all FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)


### Creamos funciones auxiliares

In [730]:
# Estas están pensadas ejecutar solo dentro de la función main

def series_sin_prof_princ(df, series, probabilidad_ppppp):
    series_sin_prof_princ = 0
    for s in series:
        df_s = df[df['Series ID'] == s]
        if len(df_s) == 0:
            # print("Serie sin asignaciones: ", s)
            continue
        else:
            prof_princ = profesor_principal(df, s)
            mode_length = len(df_s[df_s['Teacher Assigned'] == prof_princ] )
            len_dfs = len(df_s)
            percentage = str(round(100*mode_length/len_dfs,2)) + "%"
            if mode_length/len_dfs < probabilidad_ppppp:
                # print(df_s)
                series_sin_prof_princ += 1
    return series_sin_prof_princ



def total_profesores_asignados(df, series, momentos):
    profesores_utilizados = []
    for s in series:
        for i in momentos:
            clase =  df[(df['Series ID'] == s) & (df['Moment ID'] == i)]
            if len(clase) == 0:
                # print("Serie sin asignaciones: ", s)
                continue
            else:
                profesores_utilizados.append(list(df[(df['Series ID'] == s) & (df['Moment ID'] == i)]['Teacher Assigned'])[0])
    
    return len(np.unique(profesores_utilizados))


# Function to generate classes for each series
def generate_classes(series_df, start_date):
    class_rows = []
    class_id = 1
    
    for idx, row in series_df.iterrows():
        series_name = row['Series Name']
        day1 = row['Day 1']
        day2 = row['Day 2']
        time1 = int(row['Time 1'])
        time2 = int(row['Time 2'])
        series_level = row['Level Number']
        
        # Create 16 classes (8 weeks, 2 per week)
        for week in range(8):
            # Calculate the date for Day 1 of this series in this week
            day1_date = start_date + timedelta(days=(day1 - 1) + week * 7)
            class_rows.append({
                'Class ID': f'C{class_id}',
                'Series ID': series_name,
                'Series Level': series_level,
                'Starts At': datetime(day1_date.year, day1_date.month, day1_date.day, time1)
            })
            class_id += 1
            
            # Calculate the date for Day 2 of this series in this week
            day2_date = start_date + timedelta(days=(day2 - 1) + week * 7)
            class_rows.append({
                'Class ID': f'C{class_id}',
                'Series ID': series_name,
                'Series Level': series_level,
                'Starts At': datetime(day2_date.year, day2_date.month, day2_date.day, time2)
            })
            class_id += 1
    
    return pd.DataFrame(class_rows)





# Function to generate classes for each series
def generate_classes2(series_df, start_date):
    class_rows = []
    class_id = 1
    
    for idx, row in series_df.iterrows():
        series_name = row['Series Name']
        day1 = row['Day 1']
        day2 = row['Day 2']
        time = int(row['Time'])
        teacher_assigned = row['Teacher Assigned']
        
        # Create 16 classes (8 weeks, 2 per week)
        for week in range(8):
            # Calculate the date for Day 1 of this series in this week
            day1_date = start_date + timedelta(days=(day1 - 1) + week * 7)
            class_rows.append({
                'Class ID': f'C{class_id}',
                'Series ID': series_name,
                'Teacher Assigned': teacher_assigned,
                'Starts At': datetime(day1_date.year, day1_date.month, day1_date.day, time)
            })
            class_id += 1
            
            # Calculate the date for Day 2 of this series in this week
            day2_date = start_date + timedelta(days=(day2 - 1) + week * 7)
            class_rows.append({
                'Class ID': f'C{class_id}',
                'Series ID': series_name,
                'Teacher Assigned': teacher_assigned,
                'Starts At': datetime(day2_date.year, day2_date.month, day2_date.day, time)
            })
            class_id += 1
    
    return pd.DataFrame(class_rows)

        

### Función main()

In [731]:
def main(series_df, start_date, teacher_availability, teacher_info, min_availability, capa_dolares):

    classes = generate_classes(series_df, start_date)


    ## creamos vectores/variables (profesores j, momentos i, series s)
    teachers = list(teacher_info['teacher_email'].unique())
    N = len(teachers)
    momentos_timestamp = sorted(list(classes['Starts At'].unique()))
    momentos_int = list(range(1, len(momentos_timestamp) + 1))
    momentos = [str(i) for i in momentos_int]
    # Create a dictionary mapping the ID vector to the elements of M
    diccionario_momentos = {momentos[i]: momentos_timestamp[i] for i in range(len(momentos_timestamp))}
    X_c = len(momentos)
    series = list(classes['Series ID'].unique())
    Y = len(series)
    print("Tenemos ", N, " profesores disponibles para la asignación")
    print("Tenemos ", X_c, " momentos con una o mas clases")
    print("Tenemos ", Y, " series que tenemos que asignar")

    ## Transformamos la información de costos de profesores ajustado a la matriz de disponibilidad (si no tiene disponibilidad no voy a revisar su costo)
    teacher_info['fee'] = pd.to_numeric(teacher_info['fee'], errors='coerce')
    teacher_costs = pd.DataFrame(index = teachers)
    teacher_costs['fee'] = 0
    teacher_costs['Total series'] = 0
    teacher_costs['Overachiever'] = False
    for j in teachers:
        if j in list(teacher_info['teacher_email']):
            w = teacher_info[teacher_info['teacher_email'] == j]['fee'].values[0]
            t = teacher_info[teacher_info['teacher_email'] == j]['Total series'].values[0]
            x = teacher_info[teacher_info['teacher_email'] == j]['Max series'].values[0]
            k = teacher_info[teacher_info['teacher_email'] == j]['Overachiever'].values[0]
            teacher_costs.loc[j, 'fee'] = w
            teacher_costs.loc[j, 'Total series'] = t
            teacher_costs.loc[j, 'Max series'] = x
            teacher_costs.loc[j, 'Overachiever'] = k

        # if k == True:
        #     teacher_costs.loc[j, 'fee'] = overachiever_fee

    # transformamos la información de profesores para tener una tabla de los niveles que pueden dictar


    ## A partir de tabla de horas libres de cada profesor (treacher_availability) creamos la matriz de disponibilidad, con el vector de momentos y profesores
    A = np.zeros((X_c, N))
    A = pd.DataFrame(A, index = momentos, columns = teachers)
    for j in teachers:
        for i in momentos:
            filtered_df = teacher_availability[(teacher_availability['slot_time'] == diccionario_momentos[i]) & (teacher_availability['teacher_email'] == j)]
            if not filtered_df.empty:
                A[j][i] = 1



    ## A partir de tabla de clases (classes) creamos la matriz de horarios a dictar, con el vector de series y momentos
    B = np.zeros((Y, X_c))
    B = pd.DataFrame(B, index = series, columns = momentos)
    for s in series:
        for i in momentos:
            filtered_df_2 = classes[(classes['Starts At'] == diccionario_momentos[i]) & (classes['Series ID'] == s)]
            if not filtered_df_2.empty:
                B[i][s] = 1


    ## A partir de tabla descriptiva de profesores (teacher info) y la tabla de series creamos la matriz de series que puede o no dictar cada profesor
    C = np.zeros((Y, N))
    C = pd.DataFrame(B, index = series, columns = teachers)
    for s in series:
        series_level = ((series_df[series_df['Series Name'] == s])['Level Number']).iloc[0]
        level_name = 'Level ' + str(int(series_level))
        for j in teachers:
            filtered_df_3 = (teacher_info[(teacher_info['teacher_email'] == j)][level_name])
            if not filtered_df_3.empty:
                C[j][s] = filtered_df_3

    #calculamos cantidad de clases a asignar y horas disponibles mediante las matrices. 
    total_classes_ = B.sum().sum()
    total_disponibilidad = A.sum().sum()
    suma_C = C.sum().sum()

    # Primer print para definir el status de la ejecución: comprension general del problema. Estos numeros deben coincidir con el numero de filas que tienen los inputs correspondientes
    print("Hay un total de ", total_classes_, "clases a asignar")
    print("Hay un total de ", total_disponibilidad, "horas disponibles de profesores")
    print("Hay un total de ", suma_C, "disponibilidades por nivel")
    

    # assign teachers

    assigned = []
    unassigned = []
    num = 0
    
    df = teacher_costs.copy()
    df = df.sample(frac=1)
    
    min_fee = df["fee"].min()
    max_fee = df["fee"].max()

    # Build buckets bottom-up (LOWEST fee first)
    bins = np.arange(
        start=min_fee,
        stop=max_fee + capa_dolares,
        step=capa_dolares
    )

    asignaciones = 0
    capa = 1

    for lower in bins:
        print("Capa #" + str(capa))
        capa = capa + 1
        upper = lower + capa_dolares

        temp_df = df[
            (df["fee"] >= lower) &
            (df["fee"] < upper)
        ]

        if temp_df.empty:
            continue
        
        assigned_series = set()
        
        temp_df = temp_df.sample(frac=1)
        
        # print(temp_df)

        
        for s in series:
            # Find teachers available for at least X% of the classes in the series
            available_teachers = []
            total_classes = B.loc[s, :].sum()

            for j in temp_df.index:
                availability_count = sum(int(A.loc[i, j]) * int(B.loc[s, i]) for i in A.index) 
                
                if (availability_count >= min_availability * total_classes) and (C.loc[s, j] == 1) and (temp_df.loc[j, "Total series"] < temp_df.loc[j, "Max series"]):
                    available_teachers.append((j, temp_df.loc[j, "fee"], availability_count))

            if len(available_teachers) > 0:
                asignaciones = asignaciones + 1 
                print(str(asignaciones) + " ass")
                # Find the least expensive teacher
                random.shuffle(available_teachers)
                available_teachers.sort(key=lambda x: x[1])
                selected_teacher = available_teachers[0][0]
                teacher_cost = available_teachers[0][1]
                current_series = df.loc[selected_teacher, 'Total series']
                # Assign the teacher to the available classes
                for i in momentos:
                    if B.loc[s, i] == 1 and A.loc[i, selected_teacher] == 1:
                        assigned.append({"Series ID": s, "Moment ID": i, "Teacher": selected_teacher, "Cost": teacher_cost})
                        A.loc[i, selected_teacher] = 0
                        assigned_series.add(s) 
                        df.loc[selected_teacher, 'Total series'] = current_series + 1
                        temp_df.loc[selected_teacher, 'Total series'] = current_series + 1
                    else:
                        unassigned.append({"Series ID": s, "Moment ID": i})
        
        print(assigned_series)
        series = [s for s in series if s not in assigned_series]

        if not series: # revisamos si está vacía para parar el ciclo
            print("No hay mas series por asignar")
            break

    

    # Convert assignments to DataFrames
    assigned_df = pd.DataFrame(assigned)
    unassigned_df = pd.DataFrame(unassigned)


    print("Solución encontrada. Ahora se va validara la utilidad del resultado!")
    return assigned_df, unassigned_df, df
                

### Evaluacion de resultados

In [732]:
def evaluate_solution(assigned_df, series_df, start_date, teacher_availability, teacher_info):
    """
    Evaluates the solution found in assigned_df based on the following conditions:
    1. Teachers were assigned taking into account their original availability.
    2. Teachers are not assigned to two or more classes that occur at the same moment.
    3. Only one teacher per class.
    4. Teachers were assigned to classes according to their level.
    5. We are not assigning teachers to non-existing classes.
    
    Parameters:
    - assigned_df: DataFrame with the assigned classes to teachers.
    - A: Original availability matrix (moments x teachers).
    - B: Original series schedule matrix (series x moments).
    - C: Original teacher suitability matrix (series x teachers).
    
    Returns:
    - evaluation_results: Dictionary containing DataFrames with classes that do not meet each condition.
    """

    classes = generate_classes(series_df, start_date)


    ## creamos vectores/variables (profesores j, momentos i, series s)
    teachers = list(teacher_info['teacher_email'].unique())
    N = len(teachers)
    momentos_timestamp = sorted(list(classes['Starts At'].unique()))
    momentos_int = list(range(1, len(momentos_timestamp) + 1))
    momentos = [str(i) for i in momentos_int]
    # Create a dictionary mapping the ID vector to the elements of M
    diccionario_momentos = {momentos[i]: momentos_timestamp[i] for i in range(len(momentos_timestamp))}
    X_c = len(momentos)
    series = list(classes['Series ID'].unique())
    Y = len(series)


    ## Transformamos la información de costos de profesores ajustado a la matriz de disponibilidad (si no tiene disponibilidad no voy a revisar su costo)
    teacher_info['fee'] = pd.to_numeric(teacher_info['fee'], errors='coerce')
    teacher_costs = pd.DataFrame(index = teachers)
    teacher_costs['fee'] = 0
    for j in teachers:
        if j in list(teacher_info['teacher_email']):
            w = teacher_info[teacher_info['teacher_email'] == j]['fee'].values[0]
            teacher_costs.loc[j] = w

    # transformamos la información de profesores para tener una tabla de los niveles que pueden dictar


    ## A partir de tabla de horas libres de cada profesor (treacher_availability) creamos la matriz de disponibilidad, con el vector de momentos y profesores
    A = np.zeros((X_c, N))
    A = pd.DataFrame(A, index = momentos, columns = teachers)
    for j in teachers:
        for i in momentos:
            filtered_df = teacher_availability[(teacher_availability['slot_time'] == diccionario_momentos[i]) & (teacher_availability['teacher_email'] == j)]
            if not filtered_df.empty:
                print(1)
                A[j][i] = 1


    ## A partir de tabla de clases (classes) creamos la matriz de horarios a dictar, con el vector de series y momentos
    B = np.zeros((Y, X_c))
    B = pd.DataFrame(B, index = series, columns = momentos)
    for s in series:
        for i in momentos:
            filtered_df_2 = classes[(classes['Starts At'] == diccionario_momentos[i]) & (classes['Series ID'] == s)]
            if not filtered_df_2.empty:
                B[i][s] = 1


    ## A partir de tabla descriptiva de profesores (teacher info) y la tabla de series creamos la matriz de series que puede o no dictar cada profesor
    C = np.zeros((Y, N))
    C = pd.DataFrame(B, index = series, columns = teachers)
    for s in series:
        series_level = ((series_df[series_df['Series Name'] == s])['Level Number']).iloc[0]
        level_name = 'Level ' + str(series_level)
        for j in teachers:
            filtered_df_3 = (teacher_info[(teacher_info['teacher_email'] == j)][level_name])
            if not filtered_df_3.empty:
                C[j][s] = filtered_df_3
    
    # 1. Teachers were assigned taking into account their original availability
    availability_issues = assigned_df[
        assigned_df.apply(lambda row: A.loc[row['Moment ID'], row['Teacher']] != 1, axis=1)
    ]
    
    # 2. Teachers are not assigned to two or more classes that occur at the same moment
    simultaneous_issues = assigned_df.groupby(['Moment ID', 'Teacher']).filter(lambda x: len(x) > 1)
    
    # 3. Only one teacher per class
    unique_teacher_issues = assigned_df.groupby(['Series ID', 'Moment ID']).filter(lambda x: len(x) > 1)
    
    # 4. Teachers were assigned to classes according to their level
    level_issues = assigned_df[
        assigned_df.apply(lambda row: C.loc[row['Series ID'], row['Teacher']] != 1, axis=1)
    ]
    
    # 5. We are not assigning teachers to non-existing classes
    non_existing_classes_issues = assigned_df[
        assigned_df.apply(lambda row: B.loc[row['Series ID'], row['Moment ID']] != 1, axis=1)
    ]

    evaluation_results = {
        "Availability_Issues": availability_issues,
        "Simultaneous_Issues": simultaneous_issues,
        "Unique_Teacher_Issues": unique_teacher_issues,
        "Level_Issues": level_issues,
        "Non_Existing_Classes_Issues": non_existing_classes_issues
    }
    
    return evaluation_results




In [733]:
def calculate_main_teacher_assistance(assigned_df):
    # Count the number of classes each teacher is assigned to in each series
    series_teacher_counts = (
        assigned_df.groupby(['Series ID', 'Teacher'])
        .size()
        .reset_index(name='Assigned Classes Count')
    )
    
    # Find the main teacher for each series (teacher with the most classes)
    main_teachers = (
        series_teacher_counts.loc[series_teacher_counts.groupby('Series ID')['Assigned Classes Count'].idxmax()]
        .reset_index(drop=True)
    )
    
    # Calculate the percentage of classes the main teacher is assigned to
    total_classes_per_series = assigned_df['Series ID'].value_counts().reset_index()
    total_classes_per_series.columns = ['Series ID', 'Total Classes']

    total_classes_per_series['Total Classes'] = 16

    main_teacher_summary = pd.merge(main_teachers, total_classes_per_series, on='Series ID')
    main_teacher_summary['Percentage of Assistance'] = (
        main_teacher_summary['Assigned Classes Count'] / main_teacher_summary['Total Classes']
    ) * 100




    return main_teacher_summary


### Importamos input

In [734]:
start_date = datetime(2026, 1, 12)
series_df = pd.read_csv('new_series.csv')
teacher_availability = pd.read_csv('teacher_availability_matrix.csv')

In [735]:
teacher_availability

,date_period,start_time_period,end_time_period,email_prof,portuguese,spanish,native,recurring_availability,calendar_class,teach_app_series,series_id,c_type,t_type,availability,scheduled_hours,available_hours,occupancy_ratio,class_offering,weekday,hour
0,2025-12-29,05:00:00,06:00:00,j.klawa@slangapp.com,True,False,False,Yes,NaN,NaN,0,NaN,NaN,Yes,NaN,0,NaN,NaN,1,5
1,2026-01-05,05:00:00,06:00:00,j.klawa@slangapp.com,True,False,False,Yes,NaN,NaN,0,NaN,NaN,Yes,NaN,0,NaN,NaN,1,5
2,2026-01-12,05:00:00,06:00:00,j.klawa@slangapp.com,True,False,False,Yes,NaN,NaN,0,NaN,NaN,Yes,NaN,0,NaN,NaN,1,5
3,2026-01-19,05:00:00,06:00:00,j.klawa@slangapp.com,True,False,False,Yes,NaN,NaN,0,NaN,NaN,Yes,NaN,0,NaN,NaN,1,5
4,2026-01-26,05:00:00,06:00:00,j.klawa@slangapp.com,True,False,False,Yes,NaN,NaN,0,NaN,NaN,Yes,NaN,0,NaN,NaN,1,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69717,2026-06-13,21:00:00,22:00:00,g.vega@slangapp.com,False,True,False,Yes,NaN,NaN,0,NaN,NaN,Yes,NaN,0,NaN,NaN,6,21
69718,2026-06-06,21:00:00,22:00:00,g.vega@slangapp.com,False,True,False,Yes,NaN,NaN,0,NaN,NaN,Yes,NaN,0,NaN,NaN,6,21
69719,2026-06-20,21:00:00,22:00:00,g.vega@slangapp.com,False,True,False,Yes,NaN,NaN,0,NaN,NaN,Yes,NaN,0,NaN,NaN,6,21
69720,2026-06-27,21:00:00,22:00:00,g.vega@slangapp.com,False,True,False,Yes,NaN,NaN,0,NaN,NaN,Yes,NaN,0,NaN,NaN,6,21


In [736]:
teacher_availability = teacher_availability[['email_prof','date_period','start_time_period']]
teacher_availability['slot_time'] =  pd.to_datetime(teacher_availability['date_period'] + ' ' + teacher_availability['start_time_period'])
teacher_availability = teacher_availability[['email_prof', 'slot_time']]
teacher_availability['teacher_email'] = teacher_availability['email_prof']
teacher_availability = teacher_availability[['slot_time', 'teacher_email']]
teacher_availability

/var/folders/_6/jl4t8d4x7hj0k2ct_kt81_3w0000gn/T/ipykernel_27334/3005643807.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  teacher_availability['slot_time'] =  pd.to_datetime(teacher_availability['date_period'] + ' ' + teacher_availability['start_time_period'])


,slot_time,teacher_email
0,2025-12-29 05:00:00,j.klawa@slangapp.com
1,2026-01-05 05:00:00,j.klawa@slangapp.com
2,2026-01-12 05:00:00,j.klawa@slangapp.com
3,2026-01-19 05:00:00,j.klawa@slangapp.com
4,2026-01-26 05:00:00,j.klawa@slangapp.com
...,...,...
69717,2026-06-13 21:00:00,g.vega@slangapp.com
69718,2026-06-06 21:00:00,g.vega@slangapp.com
69719,2026-06-20 21:00:00,g.vega@slangapp.com
69720,2026-06-27 21:00:00,g.vega@slangapp.com


In [737]:
classes = generate_classes(series_df, start_date)
momentos = sorted(list(classes['Starts At'].unique()))
X_c = len(momentos)

In [738]:

teacher_info = pd.read_csv('teachers.csv')
teacher_info.head()

,teacher_email,Level 1,Level 2,Level 3,Level 4,Level 5,Level 6,Level 7,Level 8,Level 9,...,Practice,Premium,Executive,PIT,fee,Type,Tier,Avg Hrs Assigned (Aug–Oct),Current Hours,Max series
0,l.mamani@slangapp.com,False,True,True,True,True,True,True,True,True,...,True,True,True,False,7.0,Part Time,1,100,17,14
1,j.klawa@slangapp.com,True,True,True,True,True,True,True,True,True,...,True,True,True,False,8.0,Part Time,1,124,6,15
2,k.gonzalez@slangapp.com,True,True,True,True,True,True,True,True,True,...,True,True,True,False,8.0,Part Time,1,83,17,14
3,s.rodriguez@slangapp.com,True,True,True,True,True,True,True,True,True,...,True,True,True,False,8.0,Part Time,1,42,0,16
4,m.donato@slangapp.com,True,True,True,True,True,True,True,True,True,...,True,True,True,False,8.0,Part Time,1,42,0,16


In [739]:

teacher_info = teacher_info[['teacher_email', 'Level 1', 'Level 2', 'Level 3', 'Level 4', 'Level 5', 'Level 6', 'Level 7', 'Level 8', 'Level 9', 'Level 10', 'Level 11', 'Level 12', 'Level 13', 'Level 14', 'Level 15', 'Level 16', 'fee', 'Max series']]
teacher_info['Total series'] = 0
teacher_info['Overachiever'] = False
teacher_info.head()

,teacher_email,Level 1,Level 2,Level 3,Level 4,Level 5,Level 6,Level 7,Level 8,Level 9,...,Level 11,Level 12,Level 13,Level 14,Level 15,Level 16,fee,Max series,Total series,Overachiever
0,l.mamani@slangapp.com,False,True,True,True,True,True,True,True,True,...,False,False,False,False,False,False,7.0,14,0,False
1,j.klawa@slangapp.com,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,8.0,15,0,False
2,k.gonzalez@slangapp.com,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,8.0,14,0,False
3,s.rodriguez@slangapp.com,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,8.0,16,0,False
4,m.donato@slangapp.com,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,8.0,16,0,False


In [740]:

# Convert only boolean columns to int
object_dtype_cols = teacher_info.select_dtypes(include='object').columns
#manejamos errores! Valores nulos, etc. 
for col in teacher_info.columns:
    if col not in (['teacher_email', 'fee']):
        teacher_info[col] = teacher_info[col].fillna(False)
        teacher_info[col] = teacher_info[col].astype(int)

start_date = datetime(2026, 1, 12)

In [741]:
teacher_availability.head()

,slot_time,teacher_email
0,2025-12-29 05:00:00,j.klawa@slangapp.com
1,2026-01-05 05:00:00,j.klawa@slangapp.com
2,2026-01-12 05:00:00,j.klawa@slangapp.com
3,2026-01-19 05:00:00,j.klawa@slangapp.com
4,2026-01-26 05:00:00,j.klawa@slangapp.com


In [742]:
teacher_info.tail()

,teacher_email,Level 1,Level 2,Level 3,Level 4,Level 5,Level 6,Level 7,Level 8,Level 9,...,Level 11,Level 12,Level 13,Level 14,Level 15,Level 16,fee,Max series,Total series,Overachiever
61,e.beltran@slangapp.com,0,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,10.5,14,0,0
62,g.vega@slangapp.com,1,1,1,1,1,1,1,1,1,...,1,1,0,0,0,0,10.7,10,0,0
63,c.urrutia@slangapp.com,0,1,1,1,1,1,1,1,1,...,1,0,0,0,0,0,10.8,14,0,0
64,c.ardila@slangapp.com,0,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,12.0,15,0,0
65,r.caballero@slangapp.com,0,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,12.0,15,0,0


### Ejecucion

In [743]:
series_df[(series_df['Level Number'].isnull())]

,Series ID,Series Name,Group ID,New Level,Level Number,Day 1,Day 2,Start Hours (UTC-5),Time 1,Time 2,Start Date,End Date


In [751]:
# series_df = series_df[:50]
assigned_df, unassigned_df, teacher_costs = main(series_df, start_date, teacher_availability, teacher_info, 1, 0.5)
# df, teacher_costs_filtered = main(series_df, start_date, teacher_availability, teacher_info, 1, 0.5) 
# cerca de 2 minutos de ejcucion 

Tenemos  66  profesores disponibles para la asignación
Tenemos  624  momentos con una o mas clases
Tenemos  299  series que tenemos que asignar
Hay un total de  4784.0 clases a asignar
Hay un total de  13968.0 horas disponibles de profesores
Hay un total de  17915.0 disponibilidades por nivel
Capa #1
1 ass
2 ass
3 ass
4 ass
5 ass
6 ass
7 ass
{'Online Classes - Tue & Fri - 125', 'Online Classes - Wed & Fri - 138', 'Online Classes - Tue & Wed - 140', 'Online Classes - Wed & Fri - 95', 'Online Classes - Fri & Sat - 11', 'Online Classes - Tue & Wed - 77', 'Online Classes - Mon & Tue - 139'}
Capa #2
Capa #3
8 ass
9 ass
10 ass
11 ass
12 ass
13 ass
14 ass
15 ass
16 ass
17 ass
18 ass
19 ass
20 ass
21 ass
22 ass
23 ass
24 ass
25 ass
26 ass
27 ass
28 ass
29 ass
30 ass
31 ass
32 ass
33 ass
34 ass
35 ass
36 ass
37 ass
38 ass
39 ass
40 ass
41 ass
42 ass
43 ass
44 ass
45 ass
46 ass
47 ass
48 ass
49 ass
50 ass
51 ass
52 ass
53 ass
54 ass
55 ass
56 ass
57 ass
58 ass
59 ass
60 ass
61 ass
62 ass
63 ass


In [752]:
unassigned_df.to_csv('non_assigned_series.csv')

In [753]:
print('Costo Promedio! $', round(np.mean(assigned_df['Cost']),2))

Costo Promedio! $ 8.83


In [725]:
main_teachers = calculate_main_teacher_assistance(assigned_df)
main_teachers

,Series ID,Teacher,Assigned Classes Count,Total Classes,Percentage of Assistance
0,Online Classes - Fri & Sat - 11,l.mamani@slangapp.com,16,16,100.0
1,Online Classes - Fri & Sat - 12,j.klawa@slangapp.com,16,16,100.0
2,Online Classes - Fri & Sat - 133,j.klawa@slangapp.com,16,16,100.0
3,Online Classes - Fri & Sat - 151,a.jaramillo@slangapp.com,16,16,100.0
4,Online Classes - Fri & Sat - 193,j.klawa@slangapp.com,16,16,100.0
...,...,...,...,...,...
220,Online Classes - Zurich - G3,a.siqueira@slangapp.com,16,16,100.0
221,Online Classes - Zurich - G4,g.vega@slangapp.com,16,16,100.0
222,Online Classes - Zurich - G5,li.ramirez@slangapp.com,16,16,100.0
223,Online Classes - Zurich - G6,c.franco@slangapp.com,16,16,100.0


In [726]:
main_teachers.to_csv('asignaciones_con_PA%.csv') #queremos estar cerca de las 170 series asignadas, ojalá :pray: 

In [727]:
len(main_teachers)

225